# GeoSquare starter

Load one cell dataset, roll it up to a coarser level with `df.gsq`, draw a map, and export a CSV.

- Run this notebook from the `notebooks` folder. The sample is in `sample/`.
- It needs the `gsq` accessor from geosquare-engine (main, after PR #39). The PyPI release 0.2.0 does not have it yet.
- It needs matplotlib: `uv sync --extra test --extra notebook`.
- The sample is synthetic. It is not real data.

In [ ]:
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon

import geosquare_engine.pandas_gsq  # noqa: F401, adds the df.gsq accessor
import pandas as pd
from geosquare_v2 import read_cell_dataset

from geosquare_platform.cellstore import Governance, ValueField, store_csv
from geosquare_platform.registry import load_registry

LEVEL = 11  # roll the level 12 cells up to 100 m cells

## 1. Load a dataset

The platform stores each upload as a cell dataset. This step stores the sample CSV in a temporary folder, the same way an upload does. Then it reads the dataset back and checks its manifest.

In [ ]:
registry = load_registry()
work = Path(tempfile.mkdtemp(prefix='gsq_starter_'))
fields = [
    ValueField('people', 'numeric', 'COUNT', None, 'sum', None),
    ValueField('income_index', 'numeric', 'MEASUREMENT', 'index', 'median', None),
]
stored = store_csv(
    registry,
    work / 'cells',
    Path('sample/sample_cells.csv').read_bytes(),
    value_fields=fields,
    governance=Governance(license='synthetic sample, not real data', share_policy='public'),
)
dataset = read_cell_dataset(stored.path, registry=registry)
df = dataset.table.to_pandas()
print('dataset', stored.dataset_id, 'level', stored.level, 'rows', len(df))
df.head()

## 2. Roll up with df.gsq

`df.gsq.parent(11)` gives each cell's parent at level 11. The sum follows the `COUNT` rule in the manifest. The `income_index` median is not rolled up here.

In [ ]:
df['parent'] = df.gsq.parent(LEVEL)
rolled = df.groupby('parent', as_index=False).agg(people=('people', 'sum'), cells=('gid', 'count'))
rolled

## 3. Draw the parent cells

`to_geo` gives a GeoJSON text per cell. Matplotlib draws them.

In [ ]:
domain_id = registry.get('ID').domain_id
geo = rolled.gsq.to_geo(domain_id=domain_id, column='parent')
shapes = [Polygon(json.loads(text)['coordinates'][0]) for text in geo['geometry']]

fig, ax = plt.subplots(figsize=(7, 6))
collection = PatchCollection(shapes, edgecolor='black', linewidth=0.3)
collection.set_array(geo['people'].to_numpy(dtype=float))
ax.add_collection(collection)
ax.autoscale_view()
ax.set_aspect('equal')
ax.set_title(f'People per level {LEVEL} cell (synthetic)')
fig.colorbar(collection, ax=ax, label='people')
plt.show()

## 4. Export

Write the rolled-up table to CSV.

In [ ]:
out = work / f'rolled_level_{LEVEL}.csv'
rolled.to_csv(out, index=False)
print('wrote', out)
pd.read_csv(out).head()